In [ ]:
"""
================================================================================
SECURITY INVESTIGATION PLAYBOOK (Delta Tables - Optimized)
================================================================================
Investigate audit events using Delta tables with Z-ORDER optimization.
Much faster than Parquet queries!
================================================================================
"""

from pyspark.sql.functions import *

# =============================================================================
# CONFIGURATION
# =============================================================================

EVENT_NAME = "CreatePrivateIp"
PARTITION_DATE = "2026-01-28"

# Limit results to avoid memory issues
MAX_RESULTS = 50

# =============================================================================
# DELTA TABLES (Fast!)
# =============================================================================
SILVER_AUDIT_TABLE = "gitrepo.default.silver_audit_logs"
SILVER_FLOW_TABLE = "gitrepo.default.silver_flow_logs"

print("=" * 80)
print(f"SECURITY INVESTIGATION: {EVENT_NAME}")
print("=" * 80)
print(f"  Partition: {PARTITION_DATE}")
print(f"  Using Delta Tables (Z-ORDER optimized)")
print("=" * 80)

# =============================================================================
# PHASE 1: INITIAL TRIAGE
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 1: INITIAL TRIAGE")
print("=" * 80)

# Load from Delta table with partition filter
silver_audit = spark.table(SILVER_AUDIT_TABLE) \
    .filter(col("ingest_date") == PARTITION_DATE)

# 1a. Quick count of target events
print("\n--- 1a. Event Count ---")
target_count = silver_audit.filter(col("event_name") == EVENT_NAME).count()
print(f"Total {EVENT_NAME} events: {target_count:,}")

if target_count == 0:
    print(f"\nNo {EVENT_NAME} events found. Top events:")
    silver_audit.groupBy("event_name").count().orderBy(desc("count")).limit(30).show(truncate=False)
    raise Exception(f"No events for {EVENT_NAME}")

# 1b. Daily volume
print("\n--- 1b. Daily Volume ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .withColumn("event_date", to_date("event_time")) \
    .groupBy("event_date") \
    .agg(count("*").alias("count")) \
    .orderBy("event_date") \
    .show(30, truncate=False)

# 1c. Success/Failure
print("\n--- 1c. Success/Failure ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("response_status") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .show(10, truncate=False)

# =============================================================================
# PHASE 2: ATTRIBUTION - Who performed these actions?
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 2: ATTRIBUTION - Who performed these actions?")
print("=" * 80)

# 2a. Principals
print("\n--- 2a. Principals (Users/Services that triggered the event) ---")
principal_df = silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("principal_id", "principal_name", "auth_type") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(MAX_RESULTS)

principal_df.show(MAX_RESULTS, truncate=False)

# Collect principal list
principal_list = [row.principal_id for row in principal_df.select("principal_id").collect() if row.principal_id]
print(f"\nPrincipals to investigate: {principal_list[:10]}...")

# 2b. Source IPs
print("\n--- 2b. Source IPs (Where requests originated from) ---")
ip_df = silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("ip_address") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(MAX_RESULTS)

ip_df.show(MAX_RESULTS, truncate=False)

# Collect IP list
ip_list = [row.ip_address for row in ip_df.select("ip_address").collect() if row.ip_address]
print(f"\nIPs to investigate: {ip_list}")

# =============================================================================
# PHASE 3: CONTEXT - What else did these actors do?
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 3: CONTEXT - What else did these actors do?")
print("=" * 80)

# 3a. All events by these principals (with OCID)
print("\n--- 3a. All Events by Same Principals ---")
print("Shows what OTHER actions these users/services performed (possible attack chain)")
if principal_list:
    silver_audit \
        .filter(col("principal_id").isin(principal_list[:5])) \
        .groupBy("principal_id", "principal_name", "event_name") \
        .agg(
            count("*").alias("count"),
            min("event_time").alias("first_seen"),
            max("event_time").alias("last_seen")
        ) \
        .orderBy(desc("count")) \
        .limit(MAX_RESULTS) \
        .show(MAX_RESULTS, truncate=False)

# 3b. All events from these IPs
print("\n--- 3b. All Events from Same IPs ---")
print("Shows what OTHER actions came from these IP addresses")
if ip_list:
    silver_audit \
        .filter(col("ip_address").isin(ip_list)) \
        .groupBy("ip_address", "event_name") \
        .agg(
            count("*").alias("count"),
            min("event_time").alias("first_seen"),
            max("event_time").alias("last_seen")
        ) \
        .orderBy(desc("count")) \
        .limit(MAX_RESULTS) \
        .show(MAX_RESULTS, truncate=False)

# 3c. Failures by these actors (ENHANCED)
print("\n--- 3c. Failed Events (Possible Attack Indicators) ---")
print("""
Failed events can indicate:
  - Brute force attempts (many auth failures)
  - Permission probing (403 Forbidden)
  - Resource enumeration (404 Not Found)
  - Rate limiting (429 Too Many Requests)
  - Misconfiguration or service issues (500 errors)
""")
if principal_list or ip_list:
    silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(~col("response_status").isin(["200", "201", "202", "204"])) \
        .groupBy("event_name", "response_status", "principal_name", "ip_address") \
        .agg(
            count("*").alias("failure_count"),
            min("event_time").alias("first_failure"),
            max("event_time").alias("last_failure"),
            first("response_message").alias("error_message")
        ) \
        .orderBy(desc("failure_count")) \
        .limit(30) \
        .show(30, truncate=False)

# 3d. Timeline
print("\n--- 3d. Activity Timeline ---")
print("""
WHAT THIS SHOWS:
  Chronological sequence of ALL events by the suspicious actors.
  Use this to understand the attack sequence / kill chain:
    1. Initial access (first events)
    2. Reconnaissance (List*, Get* events)
    3. Privilege escalation (IAM changes)
    4. Persistence (CreatePrivateIp, CreatePar, etc.)
    5. Lateral movement (network changes)
""")
if principal_list or ip_list:
    silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .select(
            "event_time", 
            "event_name", 
            "principal_name",
            "principal_id",
            "ip_address", 
            "response_status",
            "compartment_name",
            "resource_name"
        ) \
        .orderBy("event_time") \
        .limit(50) \
        .show(50, truncate=False)

# =============================================================================
# PHASE 4: IMPACT - What resources were affected?
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 4: IMPACT - What resources were affected?")
print("=" * 80)

print("\n--- 4a. Compartments Affected ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("compartment_name") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count")) \
    .limit(20) \
    .show(20, truncate=False)

# =============================================================================
# PHASE 5: CORRELATION (Kill Chain Detection)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 5: CORRELATION - Kill Chain Detection")
print("=" * 80)

# 5a. Network events
print("\n--- 5a. Network Changes (Lateral Movement Indicators) ---")
if principal_list or ip_list:
    network = silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(
            col("event_name").contains("Security") |
            col("event_name").contains("Route") |
            col("event_name").contains("Gateway") |
            col("event_name").contains("Vcn") |
            col("event_name").contains("Subnet")
        ) \
        .groupBy("event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count"))
    
    if network.count() > 0:
        print("⚠️ Network events found:")
        network.show(20, truncate=False)
    else:
        print("✅ No network changes")

# 5b. IAM events
print("\n--- 5b. IAM Changes (Privilege Escalation Indicators) ---")
if principal_list or ip_list:
    iam = silver_audit \
        .filter(
            (col("principal_id").isin(principal_list[:5])) | 
            (col("ip_address").isin(ip_list))
        ) \
        .filter(
            col("event_name").contains("User") |
            col("event_name").contains("Policy") |
            col("event_name").contains("Group") |
            col("event_name").contains("ApiKey")
        ) \
        .groupBy("event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count"))
    
    if iam.count() > 0:
        print("🚨 IAM events found:")
        iam.show(20, truncate=False)
    else:
        print("✅ No IAM changes")

# =============================================================================
# PHASE 6: FLOW LOGS - Network Traffic Analysis
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 6: FLOW LOGS - Network Traffic Analysis")
print("=" * 80)
print("""
WHAT ARE FLOW LOGS?
  Flow logs capture network traffic metadata (NOT packet contents):
  - Source IP & Port: Where traffic originated
  - Destination IP & Port: Where traffic was going  
  - Protocol: TCP, UDP, ICMP
  - Action: ACCEPT (allowed) or REJECT (blocked by security rules)
  - Bytes/Packets: Volume of data transferred
  - VNIC: The virtual network interface card in OCI

WHY THIS MATTERS FOR INVESTIGATION:
  - See if suspicious IPs communicated with your infrastructure
  - Identify data exfiltration (large outbound transfers)
  - Detect port scanning (many rejected connections)
  - Find lateral movement (internal IP to internal IP traffic)
  - Correlate audit events with actual network activity
""")

print(f"\nInvestigating IPs: {ip_list}")

# Load flow logs from Delta table
silver_flow = spark.table(SILVER_FLOW_TABLE) \
    .filter(col("ingest_date") == PARTITION_DATE)

# 6a. Traffic FROM these IPs (Outbound from suspicious sources)
print("\n--- 6a. Traffic FROM Suspicious IPs (Outbound) ---")
print("Shows where these IPs sent traffic TO (potential C2, exfiltration)")
if ip_list:
    flow_from = silver_flow \
        .filter(col("src_ip").isin(ip_list)) \
        .groupBy("src_ip", "dst_ip", "dst_port", "protocol_name", "action") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            sum("packets").alias("total_packets"),
            min("event_time").alias("first_seen"),
            max("event_time").alias("last_seen")
        ) \
        .withColumn("total_mb", round(col("total_bytes") / (1024 * 1024), 2)) \
        .orderBy(desc("total_bytes")) \
        .limit(30)
    
    if flow_from.count() > 0:
        flow_from.show(30, truncate=False)
    else:
        print("""
No outbound flows found. Possible reasons:
  - These IPs are OCI internal service IPs (not captured in VCN flow logs)
  - These are external API endpoints calling INTO your environment
  - Flow logs not enabled for these subnets
  - Different date range than audit events
""")

# 6b. Traffic TO these IPs (Inbound to suspicious destinations)
print("\n--- 6b. Traffic TO Suspicious IPs (Inbound) ---")
print("Shows what systems connected TO these IPs (potential victims, sources)")
if ip_list:
    flow_to = silver_flow \
        .filter(col("dst_ip").isin(ip_list)) \
        .groupBy("src_ip", "dst_ip", "dst_port", "protocol_name", "action") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            sum("packets").alias("total_packets"),
            min("event_time").alias("first_seen"),
            max("event_time").alias("last_seen")
        ) \
        .withColumn("total_mb", round(col("total_bytes") / (1024 * 1024), 2)) \
        .orderBy(desc("total_bytes")) \
        .limit(30)
    
    if flow_to.count() > 0:
        flow_to.show(30, truncate=False)
    else:
        print("No inbound flows found to these IPs.")

# 6c. VNIC Details - Which network interfaces were involved?
print("\n--- 6c. VNIC and Subnet Details ---")
print("""
VNIC = Virtual Network Interface Card
  Each compute instance has one or more VNICs attached.
  This shows which specific network interfaces handled traffic.
""")
if ip_list:
    vnic_details = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .groupBy("vnic_id", "subnet_id", "src_ip", "dst_ip", "dst_port") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            first("action").alias("action"),
            min(to_date("event_time")).alias("first_date"),
            max(to_date("event_time")).alias("last_date")
        ) \
        .orderBy(desc("flow_count")) \
        .limit(30)
    
    if vnic_details.count() > 0:
        vnic_details.show(30, truncate=False)
    else:
        print("No VNIC details found for these IPs.")

# 6d. Daily flow summary for suspicious IPs
print("\n--- 6d. Daily Traffic Summary for Suspicious IPs ---")
print("Shows traffic volume by day - useful for identifying attack timeline")
if ip_list:
    daily_flows = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .withColumn("flow_date", to_date("event_time")) \
        .groupBy("flow_date") \
        .agg(
            count("*").alias("total_flows"),
            sum("bytes").alias("total_bytes"),
            countDistinct("src_ip").alias("unique_src_ips"),
            countDistinct("dst_ip").alias("unique_dst_ips"),
            countDistinct("dst_port").alias("unique_ports"),
            sum(when(col("action") == "REJECT", 1).otherwise(0)).alias("rejected_flows")
        ) \
        .withColumn("total_mb", round(col("total_bytes") / (1024 * 1024), 2)) \
        .orderBy("flow_date")
    
    if daily_flows.count() > 0:
        daily_flows.show(30, truncate=False)
    else:
        print("No daily flow data found.")

# 6e. Rejected connections (blocked by security rules)
print("\n--- 6e. Rejected Connections (Blocked Traffic) ---")
print("""
Rejected flows indicate:
  - Port scanning attempts
  - Blocked attack attempts  
  - Misconfigured security lists
  - Attempted access to restricted resources
""")
if ip_list:
    rejected = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .filter(col("action") == "REJECT") \
        .groupBy("src_ip", "dst_ip", "dst_port", "protocol_name") \
        .agg(
            count("*").alias("reject_count"),
            min("event_time").alias("first_attempt"),
            max("event_time").alias("last_attempt")
        ) \
        .orderBy(desc("reject_count")) \
        .limit(30)
    
    if rejected.count() > 0:
        print("⚠️ Blocked traffic found:")
        rejected.show(30, truncate=False)
    else:
        print("✅ No rejected connections found for these IPs.")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("INVESTIGATION SUMMARY")
print("=" * 80)
print(f"""
TARGET EVENT:     {EVENT_NAME}
EVENT COUNT:      {target_count:,}
PARTITION DATE:   {PARTITION_DATE}

ACTORS IDENTIFIED:
  Principals:     {len(principal_list)} unique users/services
  Source IPs:     {len(ip_list)} unique IP addresses

IPs TO INVESTIGATE:
  {ip_list}

NEXT STEPS:
  1. Review Timeline (3d) for attack sequence
  2. Check Failed Events (3c) for brute force/probing
  3. Investigate any IAM/Network changes (5a, 5b)
  4. Correlate with flow logs for data exfiltration
  5. Check if IPs are known bad actors (threat intel)
""")

print("=" * 80)
print("Investigation complete.")
print("=" * 80)

SECURITY INVESTIGATION: CreatePrivateIp
  Partition: 2026-01-28
  Using Delta Tables (Z-ORDER optimized)

PHASE 1: INITIAL TRIAGE



--- 1a. Event Count ---


Total CreatePrivateIp events: 20,146

--- 1b. Daily Volume ---


+----------+-----+
|event_date|count|
+----------+-----+
|2026-01-06|1    |
|2026-01-07|32   |
|2026-01-15|9624 |
|2026-01-16|5817 |
|2026-01-17|4478 |
|2026-01-20|1    |
|2026-01-23|192  |
|2026-01-27|1    |
+----------+-----+


--- 1c. Success/Failure ---


+---------------+-----+
|response_status|count|
+---------------+-----+
|409            |19139|
|200            |1007 |
+---------------+-----+


PHASE 2: ATTRIBUTION - Who performed these actions?

--- 2a. Principals (Users/Services that triggered the event) ---


...


No outbound flows found. Possible reasons:
  - These IPs are OCI internal service IPs (not captured in VCN flow logs)
  - These are external API endpoints calling INTO your environment
  - Flow logs not enabled for these subnets
  - Different date range than audit events


--- 6b. Traffic TO Suspicious IPs (Inbound) ---
Shows what systems connected TO these IPs (potential victims, sources)


No inbound flows found to these IPs.

--- 6c. VNIC and Subnet Details ---

VNIC = Virtual Network Interface Card
  Each compute instance has one or more VNICs attached.
  This shows which specific network interfaces handled traffic.



No VNIC details found for these IPs.

--- 6d. Daily Traffic Summary for Suspicious IPs ---
Shows traffic volume by day - useful for identifying attack timeline


No daily flow data found.

--- 6e. Rejected Connections (Blocked Traffic) ---

Rejected flows indicate:
  - Port scanning attempts
  - Blocked attack attempts  
  - Misconfigured security lists
  - Attempted access to restricted resources



✅ No rejected connections found for these IPs.

INVESTIGATION SUMMARY

TARGET EVENT:     CreatePrivateIp
EVENT COUNT:      20,146
PARTITION DATE:   2026-01-28

ACTORS IDENTIFIED:
  Principals:     7 unique users/services
  Source IPs:     7 unique IP addresses

IPs TO INVESTIGATE:
  ['129.80.107.114', '10.0.0.14', '129.80.255.174', '10.0.240.159', '10.0.240.220', '10.0.1.40', '10.0.1.148']

NEXT STEPS:
  1. Review Timeline (3d) for attack sequence
  2. Check Failed Events (3c) for brute force/probing
  3. Investigate any IAM/Network changes (5a, 5b)
  4. Correlate with flow logs for data exfiltration
  5. Check if IPs are known bad actors (threat intel)

Investigation complete.


In [6]:
"""
================================================================================
ENHANCED SECURITY INVESTIGATION PLAYBOOK v2.0
================================================================================
Advanced forensic investigation with:
- Identity & Auth Context
- Baseline & Rarity Signals
- Resource Sensitivity Context
- Deeper Network Context
- Change Diff Evidence
- Temporal Correlation
- Risk Scoring
================================================================================
"""

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# =============================================================================
# CONFIGURATION
# =============================================================================

EVENT_NAME = "CreatePrivateIp"
PARTITION_DATE = "2026-01-28"
MAX_RESULTS = 50

# Time windows for correlation
ADJACENT_MINUTES_SHORT = 5
ADJACENT_MINUTES_MEDIUM = 15
ADJACENT_MINUTES_LONG = 60

# Delta Tables
SILVER_AUDIT_TABLE = "gitrepo.default.silver_audit_logs"
SILVER_FLOW_TABLE = "gitrepo.default.silver_flow_logs"

# Known internal IP ranges (customize for your environment)
INTERNAL_CIDRS = ["10.", "192.168.", "172.16.", "172.17.", "172.18.", "172.19.", 
                  "172.20.", "172.21.", "172.22.", "172.23.", "172.24.", "172.25.",
                  "172.26.", "172.27.", "172.28.", "172.29.", "172.30.", "172.31."]

# OCI Service IPs (Oracle internal)
OCI_SERVICE_IPS = ["129.213.", "134.70.", "140.91.", "147.154.", "192.29."]

# High-risk events for kill chain detection
HIGH_RISK_EVENTS = [
    # IAM - Privilege Escalation
    "CreateUser", "CreateGroup", "AddUserToGroup", "CreatePolicy", 
    "UpdatePolicy", "CreateApiKey", "UploadApiKey", "CreateAuthToken",
    "CreateCustomerSecretKey", "CreateSmtpCredential",
    # Network - Lateral Movement
    "CreateSecurityList", "UpdateSecurityList", "CreateRouteTable", 
    "UpdateRouteTable", "CreateInternetGateway", "CreateNatGateway",
    "CreateVcn", "CreateSubnet", "CreatePrivateIp",
    # Persistence
    "CreatePreauthenticatedRequest", "CreatePar", "CreateBucket",
    # Data Access
    "GetObject", "PutObject", "DeleteObject", "ListObjects",
    # Compute
    "LaunchInstance", "TerminateInstance", "CreateImage"
]

# Sensitive compartment patterns (customize)
SENSITIVE_COMPARTMENT_PATTERNS = ["prod", "production", "prd", "security", "audit", "network", "shared"]

print("=" * 80)
print(f"ENHANCED SECURITY INVESTIGATION: {EVENT_NAME}")
print("=" * 80)
print(f"  Partition: {PARTITION_DATE}")
print(f"  Adjacent Windows: ±{ADJACENT_MINUTES_SHORT}m, ±{ADJACENT_MINUTES_MEDIUM}m, ±{ADJACENT_MINUTES_LONG}m")
print("=" * 80)

# =============================================================================
# LOAD DATA
# =============================================================================

silver_audit = spark.table(SILVER_AUDIT_TABLE) \
    .filter(col("ingest_date") == PARTITION_DATE)

silver_flow = spark.table(SILVER_FLOW_TABLE) \
    .filter(col("ingest_date") == PARTITION_DATE)

# =============================================================================
# PHASE 1: INITIAL TRIAGE
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 1: INITIAL TRIAGE")
print("=" * 80)

print("\n--- 1a. Event Count ---")
target_events = silver_audit.filter(col("event_name") == EVENT_NAME)
target_count = target_events.count()
print(f"Total {EVENT_NAME} events: {target_count:,}")

if target_count == 0:
    print(f"\nNo {EVENT_NAME} events found. Top events:")
    silver_audit.groupBy("event_name").count().orderBy(desc("count")).limit(30).show(truncate=False)
    raise Exception(f"No events for {EVENT_NAME}")

print("\n--- 1b. Daily Volume with Success/Failure Breakdown ---")
silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .withColumn("event_date", to_date("event_time")) \
    .withColumn("is_success", col("response_status").isin(["200", "201", "202", "204"])) \
    .groupBy("event_date") \
    .agg(
        count("*").alias("total"),
        sum(when(col("is_success"), 1).otherwise(0)).alias("success"),
        sum(when(~col("is_success"), 1).otherwise(0)).alias("failed"),
        round(sum(when(~col("is_success"), 1).otherwise(0)) * 100.0 / count("*"), 2).alias("failure_rate_pct")
    ) \
    .orderBy("event_date") \
    .show(30, truncate=False)

# =============================================================================
# PHASE 2: IDENTITY & AUTH CONTEXT (VERY HIGH VALUE)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 2: IDENTITY & AUTH CONTEXT")
print("=" * 80)
print("""
This phase analyzes WHO performed the action and HOW they authenticated.
Key questions:
  - What type of authentication was used? (API Key, Instance Principal, OIDC, etc.)
  - Was this a human user or service/automation?
  - What credentials were used?
""")

# 2a. Principal Attribution with Auth Details
print("\n--- 2a. Principal Attribution with Auth Details ---")
principal_auth_df = target_events \
    .groupBy("principal_id", "principal_name", "auth_type", "caller_name") \
    .agg(
        count("*").alias("event_count"),
        countDistinct("ip_address").alias("unique_ips"),
        countDistinct("compartment_id").alias("compartments_accessed"),
        min("event_time").alias("first_seen"),
        max("event_time").alias("last_seen"),
        collect_set("ip_address").alias("ip_addresses")
    ) \
    .orderBy(desc("event_count")) \
    .limit(MAX_RESULTS)

principal_auth_df.show(MAX_RESULTS, truncate=False)

# Collect for later phases
principal_list = [row.principal_id for row in principal_auth_df.select("principal_id").collect() if row.principal_id]
ip_list = list(set([ip for row in principal_auth_df.select("ip_addresses").collect() for ip in (row.ip_addresses or [])]))

print(f"\nPrincipals identified: {len(principal_list)}")
print(f"IPs identified: {len(ip_list)}")

# 2b. Authentication Type Analysis
print("\n--- 2b. Authentication Type Analysis ---")
print("""
Auth Types in OCI:
  - natv: Native OCI user with password/MFA
  - api_key: API signing key (long-lived, high risk if compromised)
  - instance_principal: Instance identity (good for automation)
  - resource_principal: Resource identity
  - service_principal: OCI service identity
  - federation: Federated identity (SAML/OIDC)
""")

target_events \
    .groupBy("auth_type") \
    .agg(
        count("*").alias("event_count"),
        countDistinct("principal_id").alias("unique_principals"),
        countDistinct("ip_address").alias("unique_ips")
    ) \
    .orderBy(desc("event_count")) \
    .show(truncate=False)

# 2c. Credentials Analysis (if available)
print("\n--- 2c. Credentials Analysis ---")
print("Shows what specific credentials were used (API keys, tokens, etc.)")
target_events \
    .filter(col("credentials").isNotNull()) \
    .groupBy("principal_name", "auth_type", "credentials") \
    .agg(
        count("*").alias("usage_count"),
        min("event_time").alias("first_used"),
        max("event_time").alias("last_used")
    ) \
    .orderBy(desc("usage_count")) \
    .limit(30) \
    .show(truncate=False)

# 2d. User Agent Analysis (detect automation vs human)
print("\n--- 2d. User Agent Analysis ---")
print("""
User Agent helps identify:
  - OCI CLI: Oracle-PythonSDK/*
  - Terraform: terraform-provider-oci/*
  - Console: Mozilla/Chrome/Safari
  - Custom scripts: Python requests, curl, etc.
""")
target_events \
    .withColumn("user_agent_short", 
        when(col("user_agent").contains("terraform"), "Terraform")
        .when(col("user_agent").contains("Oracle-PythonSDK"), "OCI Python SDK")
        .when(col("user_agent").contains("Oracle-JavaSDK"), "OCI Java SDK")
        .when(col("user_agent").contains("Oracle-GoSDK"), "OCI Go SDK")
        .when(col("user_agent").contains("oci-cli"), "OCI CLI")
        .when(col("user_agent").contains("Mozilla"), "Browser/Console")
        .when(col("user_agent").contains("curl"), "curl")
        .when(col("user_agent").contains("python-requests"), "Python requests")
        .otherwise("Other/Unknown")
    ) \
    .groupBy("user_agent_short", "auth_type") \
    .agg(
        count("*").alias("event_count"),
        countDistinct("principal_id").alias("unique_principals")
    ) \
    .orderBy(desc("event_count")) \
    .show(truncate=False)

# 2e. Source IP Classification
print("\n--- 2e. Source IP Classification ---")
target_events \
    .withColumn("ip_type",
        when(col("ip_address").rlike("^10\\."), "Internal (10.x.x.x)")
        .when(col("ip_address").rlike("^192\\.168\\."), "Internal (192.168.x.x)")
        .when(col("ip_address").rlike("^172\\.(1[6-9]|2[0-9]|3[0-1])\\."), "Internal (172.16-31.x.x)")
        .when(col("ip_address").rlike("^129\\.213\\."), "OCI Service IP")
        .when(col("ip_address").rlike("^134\\.70\\."), "OCI Service IP")
        .when(col("ip_address").isNull(), "No IP (Internal Service)")
        .otherwise("External/Public")
    ) \
    .groupBy("ip_type") \
    .agg(
        count("*").alias("event_count"),
        countDistinct("ip_address").alias("unique_ips"),
        countDistinct("principal_id").alias("unique_principals")
    ) \
    .orderBy(desc("event_count")) \
    .show(truncate=False)

# =============================================================================
# PHASE 3: BASELINE & RARITY SIGNALS (HIGH VALUE)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 3: BASELINE & RARITY SIGNALS")
print("=" * 80)
print("""
This phase identifies UNUSUAL behavior:
  - First-time events (never seen before)
  - Volume anomalies (more than usual)
  - New principal+event combinations
""")

# 3a. Is this the first time these principals did this event?
print("\n--- 3a. First-Time Analysis for These Principals ---")
print("Checking if principals have done this event type before...")

# Get historical data (all dates, not just partition)
all_audit = spark.table(SILVER_AUDIT_TABLE)

principal_history = all_audit \
    .filter(col("principal_id").isin(principal_list)) \
    .filter(col("event_name") == EVENT_NAME) \
    .groupBy("principal_id", "principal_name") \
    .agg(
        min(to_date("event_time")).alias("first_ever_date"),
        max(to_date("event_time")).alias("last_seen_date"),
        count("*").alias("total_historical_count"),
        countDistinct(to_date("event_time")).alias("days_with_activity")
    ) \
    .withColumn("is_new_behavior", col("first_ever_date") == lit(PARTITION_DATE)) \
    .orderBy("first_ever_date")

principal_history.show(MAX_RESULTS, truncate=False)

new_behavior_count = principal_history.filter(col("is_new_behavior")).count()
print(f"\n🚨 NEW BEHAVIOR ALERT: {new_behavior_count} principals doing {EVENT_NAME} for FIRST TIME on {PARTITION_DATE}")

# 3b. Volume comparison to historical baseline
print("\n--- 3b. Volume Comparison to Baseline ---")
daily_baseline = all_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .withColumn("event_date", to_date("event_time")) \
    .groupBy("event_date") \
    .agg(count("*").alias("daily_count")) \
    .agg(
        avg("daily_count").alias("avg_daily"),
        stddev("daily_count").alias("stddev_daily"),
        min("daily_count").alias("min_daily"),
        max("daily_count").alias("max_daily")
    ).collect()[0]

today_count = target_count
z_score = (today_count - daily_baseline["avg_daily"]) / daily_baseline["stddev_daily"] if daily_baseline["stddev_daily"] and daily_baseline["stddev_daily"] > 0 else 0

print(f"Today's count:     {today_count:,}")
print(f"Historical avg:    {daily_baseline['avg_daily']:.1f}")
print(f"Historical stddev: {daily_baseline['stddev_daily']:.1f}")
print(f"Historical range:  {daily_baseline['min_daily']:.0f} - {daily_baseline['max_daily']:.0f}")
print(f"Z-Score:           {z_score:.2f}")

if z_score > 3:
    print(f"\n🚨 VOLUME ANOMALY: Today's count is {z_score:.1f} standard deviations above normal!")
elif z_score > 2:
    print(f"\n⚠️ ELEVATED VOLUME: Today's count is {z_score:.1f} standard deviations above normal")
else:
    print(f"\n✅ Volume is within normal range")

# 3c. Peer Comparison - Same event by others in same compartment
print("\n--- 3c. Peer Comparison ---")
print("Are other principals doing the same thing in the same compartments?")

target_compartments = [row.compartment_id for row in target_events.select("compartment_id").distinct().collect() if row.compartment_id]

silver_audit \
    .filter(col("event_name") == EVENT_NAME) \
    .filter(col("compartment_id").isin(target_compartments)) \
    .groupBy("compartment_name", "principal_name") \
    .agg(count("*").alias("event_count")) \
    .orderBy("compartment_name", desc("event_count")) \
    .show(30, truncate=False)

# =============================================================================
# PHASE 4: RESOURCE SENSITIVITY CONTEXT (HIGH VALUE)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 4: RESOURCE SENSITIVITY CONTEXT")
print("=" * 80)
print("""
Not all resources are equal. This phase identifies:
  - Production vs non-production targets
  - Security-sensitive compartments
  - Shared infrastructure
""")

# 4a. Compartment Sensitivity Analysis
print("\n--- 4a. Compartment Sensitivity Analysis ---")

# Create sensitivity flags
sensitive_pattern = "|".join(SENSITIVE_COMPARTMENT_PATTERNS)

target_events \
    .withColumn("is_sensitive", 
        lower(col("compartment_name")).rlike(sensitive_pattern)
    ) \
    .withColumn("sensitivity_level",
        when(lower(col("compartment_name")).rlike("prod|production|prd"), "🔴 PRODUCTION")
        .when(lower(col("compartment_name")).rlike("security|audit"), "🔴 SECURITY")
        .when(lower(col("compartment_name")).rlike("network|shared"), "🟡 SHARED INFRA")
        .when(lower(col("compartment_name")).rlike("dev|development|test|staging"), "🟢 NON-PROD")
        .otherwise("⚪ UNKNOWN")
    ) \
    .groupBy("sensitivity_level", "compartment_name") \
    .agg(
        count("*").alias("event_count"),
        countDistinct("principal_id").alias("unique_principals"),
        countDistinct("resource_id").alias("unique_resources")
    ) \
    .orderBy("sensitivity_level", desc("event_count")) \
    .show(30, truncate=False)

# 4b. Resource Details
print("\n--- 4b. Affected Resources ---")
target_events \
    .groupBy("compartment_name", "resource_name", "resource_id") \
    .agg(
        count("*").alias("event_count"),
        first("principal_name").alias("principal"),
        min("event_time").alias("first_event"),
        max("event_time").alias("last_event")
    ) \
    .orderBy(desc("event_count")) \
    .limit(30) \
    .show(truncate=False)

# =============================================================================
# PHASE 5: CHANGE DIFF EVIDENCE (VERY HIGH VALUE)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 5: CHANGE DIFF EVIDENCE")
print("=" * 80)
print("""
For state-changing events, shows WHAT EXACTLY changed:
  - Previous state vs Current state
  - Security rule changes
  - Policy modifications
""")

# 5a. State Changes (Before/After)
print("\n--- 5a. State Changes (Before/After) ---")
state_changes = target_events \
    .filter(
        col("state_previous").isNotNull() | 
        col("state_current").isNotNull()
    ) \
    .select(
        "event_time",
        "event_name",
        "principal_name",
        "resource_name",
        "state_previous",
        "state_current"
    ) \
    .orderBy("event_time")

if state_changes.count() > 0:
    print("⚠️ State changes detected:")
    state_changes.show(20, truncate=100)
else:
    print(f"No state change data captured for {EVENT_NAME}")

# 5b. Additional Details (often contains specifics of changes)
print("\n--- 5b. Additional Details ---")
print("These often contain specific parameters of the action:")
target_events \
    .filter(col("additional_details").isNotNull()) \
    .select(
        "event_time",
        "principal_name",
        "additional_details"
    ) \
    .limit(10) \
    .show(truncate=200)

# =============================================================================
# PHASE 6: TEMPORAL CORRELATION (HIGH VALUE)
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 6: TEMPORAL CORRELATION")
print("=" * 80)
print(f"""
Analyzing event adjacency to identify attack chains.
Time windows: ±{ADJACENT_MINUTES_SHORT}m, ±{ADJACENT_MINUTES_MEDIUM}m, ±{ADJACENT_MINUTES_LONG}m

Attack patterns are TIGHT in time - admin work is spread out.
""")

# 6a. Activity Timeline with Kill Chain Markers
print("\n--- 6a. Full Activity Timeline with Kill Chain Analysis ---")

# Get all events by suspicious actors
all_actor_events = silver_audit.filter(
    (col("principal_id").isin(principal_list)) | 
    (col("ip_address").isin(ip_list))
)

# Add kill chain phase classification
kill_chain_events = all_actor_events \
    .withColumn("kill_chain_phase",
        when(col("event_name").rlike("(?i)list|get|describe|head"), "1_RECON")
        .when(col("event_name").rlike("(?i)user|group|policy|apikey|auth|credential|password"), "2_PRIV_ESC")
        .when(col("event_name").rlike("(?i)security|route|gateway|vcn|subnet|privateip"), "3_LATERAL")
        .when(col("event_name").rlike("(?i)create|launch|upload|put"), "4_PERSIST")
        .when(col("event_name").rlike("(?i)delete|terminate|remove"), "5_CLEANUP")
        .otherwise("0_OTHER")
    ) \
    .withColumn("is_high_risk", col("event_name").isin(HIGH_RISK_EVENTS)) \
    .select(
        "event_time",
        "kill_chain_phase",
        "event_name",
        "principal_name",
        "ip_address",
        "compartment_name",
        "response_status",
        "is_high_risk"
    ) \
    .orderBy("event_time")

print("Timeline with Kill Chain Classification:")
kill_chain_events.show(100, truncate=False)

# 6b. Event Adjacency Analysis
print("\n--- 6b. Events Within Time Windows of Target Event ---")

# For each target event, find what happened before/after
target_times = [row.event_time for row in target_events.select("event_time").limit(10).collect()]

if target_times:
    sample_time = target_times[0]
    print(f"\nAnalyzing events around sample target event at: {sample_time}")
    
    # ±5 minutes
    print(f"\n  Events within ±{ADJACENT_MINUTES_SHORT} minutes:")
    silver_audit \
        .filter(
            (col("event_time") >= sample_time - expr(f"INTERVAL {ADJACENT_MINUTES_SHORT} MINUTES")) &
            (col("event_time") <= sample_time + expr(f"INTERVAL {ADJACENT_MINUTES_SHORT} MINUTES"))
        ) \
        .filter(
            (col("principal_id").isin(principal_list)) | 
            (col("ip_address").isin(ip_list))
        ) \
        .groupBy("event_name") \
        .agg(count("*").alias("count")) \
        .orderBy(desc("count")) \
        .show(20, truncate=False)

# 6c. High-Risk Event Sequences
print("\n--- 6c. High-Risk Event Sequences ---")
print("Looking for dangerous event combinations by the same actors...")

high_risk_by_actor = silver_audit \
    .filter(
        (col("principal_id").isin(principal_list)) | 
        (col("ip_address").isin(ip_list))
    ) \
    .filter(col("event_name").isin(HIGH_RISK_EVENTS)) \
    .groupBy("principal_name", "event_name") \
    .agg(
        count("*").alias("count"),
        min("event_time").alias("first"),
        max("event_time").alias("last")
    ) \
    .orderBy("principal_name", "first")

if high_risk_by_actor.count() > 0:
    print("🚨 HIGH-RISK EVENTS BY SUSPICIOUS ACTORS:")
    high_risk_by_actor.show(50, truncate=False)
else:
    print("✅ No other high-risk events found by these actors")

# =============================================================================
# PHASE 7: FAILED EVENTS DEEP DIVE
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 7: FAILED EVENTS ANALYSIS")
print("=" * 80)
print("""
Failed events indicate:
  - 401/403: Permission denied (probing, compromised creds)
  - 404: Resource not found (enumeration)
  - 409: Conflict (race conditions, automation issues)
  - 429: Rate limited (brute force, DDoS)
  - 500+: Server errors (exploitation attempts)
""")

failed_events = silver_audit \
    .filter(
        (col("principal_id").isin(principal_list)) | 
        (col("ip_address").isin(ip_list))
    ) \
    .filter(~col("response_status").isin(["200", "201", "202", "204"]))

print("\n--- 7a. Failure Summary by Status Code ---")
failed_events \
    .groupBy("response_status") \
    .agg(
        count("*").alias("failure_count"),
        countDistinct("event_name").alias("unique_events"),
        countDistinct("principal_id").alias("unique_principals")
    ) \
    .withColumn("severity",
        when(col("response_status").isin(["401", "403"]), "🔴 AUTH FAILURE")
        .when(col("response_status") == "404", "🟡 NOT FOUND")
        .when(col("response_status") == "429", "🔴 RATE LIMITED")
        .when(col("response_status").rlike("^5"), "🟡 SERVER ERROR")
        .otherwise("⚪ OTHER")
    ) \
    .orderBy(desc("failure_count")) \
    .show(truncate=False)

print("\n--- 7b. Failed Events Detail ---")
failed_events \
    .groupBy("event_name", "response_status", "principal_name", "ip_address") \
    .agg(
        count("*").alias("failure_count"),
        min("event_time").alias("first_failure"),
        max("event_time").alias("last_failure"),
        first("response_message").alias("error_message")
    ) \
    .orderBy(desc("failure_count")) \
    .limit(30) \
    .show(truncate=False)

# =============================================================================
# PHASE 8: NETWORK FLOW ANALYSIS
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 8: NETWORK FLOW ANALYSIS")
print("=" * 80)
print(f"""
Correlating audit events with network traffic.
IPs under investigation: {ip_list}
""")

# 8a. Traffic Direction Analysis
print("\n--- 8a. Traffic Direction Analysis ---")
if ip_list:
    flow_analysis = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .withColumn("direction",
            when(col("src_ip").isin(ip_list) & ~col("dst_ip").isin(ip_list), "OUTBOUND")
            .when(~col("src_ip").isin(ip_list) & col("dst_ip").isin(ip_list), "INBOUND")
            .otherwise("INTERNAL")
        ) \
        .withColumn("dst_classification",
            when(col("dst_ip").rlike("^10\\."), "Internal 10.x")
            .when(col("dst_ip").rlike("^192\\.168\\."), "Internal 192.168.x")
            .when(col("dst_ip").rlike("^172\\.(1[6-9]|2[0-9]|3[0-1])\\."), "Internal 172.x")
            .when(col("dst_ip").rlike("^129\\.213\\."), "OCI Service")
            .otherwise("External/Internet")
        ) \
        .groupBy("direction", "dst_classification", "action") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            countDistinct("dst_port").alias("unique_ports")
        ) \
        .withColumn("total_mb", round(col("total_bytes") / (1024*1024), 2)) \
        .orderBy("direction", desc("total_bytes"))
    
    if flow_analysis.count() > 0:
        flow_analysis.show(30, truncate=False)
    else:
        print("No flow data found for these IPs")

# 8b. Port Analysis
print("\n--- 8b. Destination Port Analysis ---")
if ip_list:
    silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .groupBy("dst_port", "protocol_name", "action") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            countDistinct("src_ip").alias("unique_sources"),
            countDistinct("dst_ip").alias("unique_destinations")
        ) \
        .withColumn("service_guess",
            when(col("dst_port") == 22, "SSH")
            .when(col("dst_port") == 443, "HTTPS")
            .when(col("dst_port") == 80, "HTTP")
            .when(col("dst_port") == 3389, "RDP")
            .when(col("dst_port") == 1521, "Oracle DB")
            .when(col("dst_port") == 3306, "MySQL")
            .when(col("dst_port") == 5432, "PostgreSQL")
            .when(col("dst_port").between(6443, 6443), "Kubernetes API")
            .otherwise("Other")
        ) \
        .orderBy(desc("flow_count")) \
        .limit(30) \
        .show(truncate=False)

# 8c. VNIC Details
print("\n--- 8c. VNIC and Instance Correlation ---")
if ip_list:
    vnic_details = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .filter(col("vnic_id").isNotNull()) \
        .groupBy("vnic_id", "subnet_id", "instance_id") \
        .agg(
            count("*").alias("flow_count"),
            sum("bytes").alias("total_bytes"),
            countDistinct("src_ip").alias("unique_src"),
            countDistinct("dst_ip").alias("unique_dst"),
            min(to_date("event_time")).alias("first_date"),
            max(to_date("event_time")).alias("last_date")
        ) \
        .orderBy(desc("flow_count")) \
        .limit(20)
    
    if vnic_details.count() > 0:
        vnic_details.show(truncate=False)
    else:
        print("No VNIC details found")

# 8d. Daily Traffic Pattern
print("\n--- 8d. Daily Traffic Pattern ---")
if ip_list:
    daily_traffic = silver_flow \
        .filter(
            (col("src_ip").isin(ip_list)) | 
            (col("dst_ip").isin(ip_list))
        ) \
        .withColumn("flow_date", to_date("event_time")) \
        .withColumn("hour", hour("event_time")) \
        .groupBy("flow_date", "hour") \
        .agg(
            count("*").alias("flows"),
            sum("bytes").alias("bytes"),
            sum(when(col("action") == "REJECT", 1).otherwise(0)).alias("rejected")
        ) \
        .orderBy("flow_date", "hour")
    
    if daily_traffic.count() > 0:
        daily_traffic.show(50, truncate=False)

# =============================================================================
# PHASE 9: RISK SCORING
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 9: RISK SCORING")
print("=" * 80)
print("""
Calculating composite risk scores based on all signals:
  - Identity Risk: Auth type, first-time behavior, credential type
  - Behavior Risk: Volume anomaly, high-risk events, failures
  - Resource Risk: Sensitivity of affected compartments
  - Network Risk: External access, unusual ports, traffic volume
""")

# Calculate risk factors
identity_risk = 0
behavior_risk = 0
resource_risk = 0
network_risk = 0

# Identity Risk
api_key_usage = target_events.filter(col("auth_type") == "api_key").count()
if api_key_usage > 0:
    identity_risk += 2
if new_behavior_count > 0:
    identity_risk += 3
external_ip_count = target_events.filter(~col("ip_address").rlike("^10\\.|^192\\.168\\.|^172\\.")).count()
if external_ip_count > 0:
    identity_risk += 2

# Behavior Risk
if z_score > 3:
    behavior_risk += 3
elif z_score > 2:
    behavior_risk += 2
failed_count = failed_events.count() if 'failed_events' in dir() else 0
if failed_count > 10:
    behavior_risk += 2
high_risk_count = high_risk_by_actor.count() if 'high_risk_by_actor' in dir() else 0
if high_risk_count > 5:
    behavior_risk += 3

# Resource Risk
prod_events = target_events.filter(lower(col("compartment_name")).rlike("prod|production|prd")).count()
if prod_events > 0:
    resource_risk += 3
security_events = target_events.filter(lower(col("compartment_name")).rlike("security|audit")).count()
if security_events > 0:
    resource_risk += 3

# Network Risk (placeholder - would need flow data)
network_risk = 1  # Default low

total_risk = identity_risk + behavior_risk + resource_risk + network_risk

print(f"""
RISK SCORE BREAKDOWN:
{'='*50}
  Identity Risk:   {identity_risk}/7   {'🔴' if identity_risk >= 5 else '🟡' if identity_risk >= 3 else '🟢'}
    - API Key usage: {api_key_usage > 0}
    - First-time behavior: {new_behavior_count > 0}
    - External IP access: {external_ip_count > 0}
    
  Behavior Risk:   {behavior_risk}/8   {'🔴' if behavior_risk >= 5 else '🟡' if behavior_risk >= 3 else '🟢'}
    - Volume anomaly (z={z_score:.1f}): {z_score > 2}
    - Failed events: {failed_count}
    - High-risk event chain: {high_risk_count > 0}
    
  Resource Risk:   {resource_risk}/6   {'🔴' if resource_risk >= 4 else '🟡' if resource_risk >= 2 else '🟢'}
    - Production access: {prod_events > 0}
    - Security compartment: {security_events > 0}
    
  Network Risk:    {network_risk}/5   {'🔴' if network_risk >= 4 else '🟡' if network_risk >= 2 else '🟢'}
    (Requires flow log correlation)

{'='*50}
  TOTAL RISK:      {total_risk}/26   {'🔴 HIGH' if total_risk >= 15 else '🟡 MEDIUM' if total_risk >= 8 else '🟢 LOW'}
{'='*50}
""")

# =============================================================================
# PHASE 10: INVESTIGATION SUMMARY & RECOMMENDATIONS
# =============================================================================
print("\n" + "=" * 80)
print("PHASE 10: INVESTIGATION SUMMARY")
print("=" * 80)

print(f"""
================================================================================
TARGET EVENT:          {EVENT_NAME}
PARTITION DATE:        {PARTITION_DATE}
TOTAL EVENTS:          {target_count:,}
================================================================================

ACTORS IDENTIFIED:
  Principals:          {len(principal_list)}
  Unique IPs:          {len(ip_list)}
  
KEY FINDINGS:
  New Behavior:        {new_behavior_count} principals doing this for FIRST TIME
  Volume Z-Score:      {z_score:.2f} ({'ANOMALY' if z_score > 3 else 'Elevated' if z_score > 2 else 'Normal'})
  Failed Events:       {failed_count}
  High-Risk Events:    {high_risk_count}
  Production Impact:   {prod_events} events in prod compartments

RISK ASSESSMENT:       {total_risk}/26 - {'🔴 HIGH PRIORITY' if total_risk >= 15 else '🟡 MEDIUM PRIORITY' if total_risk >= 8 else '🟢 LOW PRIORITY'}

================================================================================
RECOMMENDED ACTIONS:
================================================================================
""")

if total_risk >= 15:
    print("""
🔴 HIGH PRIORITY - IMMEDIATE ACTION REQUIRED:
  1. CONTAIN: Consider disabling compromised principals
  2. PRESERVE: Capture full logs for forensic analysis
  3. INVESTIGATE: Review all high-risk events in timeline
  4. ESCALATE: Notify security team and management
  5. REMEDIATE: Rotate any exposed credentials
""")
elif total_risk >= 8:
    print("""
🟡 MEDIUM PRIORITY - INVESTIGATE WITHIN 24 HOURS:
  1. REVIEW: Analyze timeline for suspicious patterns
  2. VERIFY: Confirm if activity is authorized
  3. MONITOR: Set up alerts for continued activity
  4. DOCUMENT: Record findings in incident tracker
""")
else:
    print("""
🟢 LOW PRIORITY - ROUTINE REVIEW:
  1. VERIFY: Confirm activity matches expected behavior
  2. DOCUMENT: Log review completion
  3. TUNE: Consider if this should trigger alerts
""")

print(f"""
================================================================================
IPs FOR THREAT INTEL LOOKUP:
{ip_list}

PRINCIPALS FOR REVIEW:
{principal_list[:5]}
================================================================================
Investigation complete.
================================================================================
""")

ENHANCED SECURITY INVESTIGATION: CreatePrivateIp
  Partition: 2026-01-28
  Adjacent Windows: ±5m, ±15m, ±60m



PHASE 1: INITIAL TRIAGE

--- 1a. Event Count ---


Total CreatePrivateIp events: 20,146

--- 1b. Daily Volume with Success/Failure Breakdown ---


+----------+-----+-------+------+----------------+
|event_date|total|success|failed|failure_rate_pct|
+----------+-----+-------+------+----------------+
|2026-01-06|1    |1      |0     |0.0             |
|2026-01-07|32   |32     |0     |0.0             |
|2026-01-15|9624 |26     |9598  |99.73           |
|2026-01-16|5817 |11     |5806  |99.81           |
|2026-01-17|4478 |743    |3735  |83.41           |
|2026-01-20|1    |1      |0     |0.0             |
|2026-01-23|192  |192    |0     |0.0             |
|2026-01-27|1    |1      |0     |0.0             |
+----------+-----+-------+------+----------------+


PHASE 2: IDENTITY & AUTH CONTEXT

This phase analyzes WHO performed the action and HOW they authenticated.
Key questions:
  - What type of authentication was used? (API Key, Instance Principal, OIDC, etc.)
  - Was this a human user or service/automation?
  - What credentials were used?


--- 2a. Principal Attribution with Auth Details ---


+----------------------------------------------------------------------------------+-------------------+---------+-----------+-----------+----------+---------------------+-----------------------+-----------------------+----------------+
|principal_id                                                                      |principal_name     |auth_type|caller_name|event_count|unique_ips|compartments_accessed|first_seen             |last_seen              |ip_addresses    |
+----------------------------------------------------------------------------------+-------------------+---------+-----------+-----------+----------+---------------------+-----------------------+-----------------------+----------------+
|ocid1.cluster.oc1.iad.aaaaaaaaaopk3ogvigo7dl5vnje4xss457pzfwvjlflhwxktmc36jjyupxha|NULL               |resource |NULL       |19726      |1         |1                    |2026-01-15 00:10:16.5  |2026-01-17 21:08:09.001|[129.80.107.114]|
|ocid1.cluster.oc1.iad.aaaaaaaa7aitgilwr2oenuvk2oqhg

...

No flow data found for these IPs

--- 8b. Destination Port Analysis ---


+--------+-------------+------+----------+-----------+--------------+-------------------+-------------+
|dst_port|protocol_name|action|flow_count|total_bytes|unique_sources|unique_destinations|service_guess|
+--------+-------------+------+----------+-----------+--------------+-------------------+-------------+
+--------+-------------+------+----------+-----------+--------------+-------------------+-------------+


--- 8c. VNIC and Instance Correlation ---


No VNIC details found

--- 8d. Daily Traffic Pattern ---



PHASE 9: RISK SCORING

Calculating composite risk scores based on all signals:
  - Identity Risk: Auth type, first-time behavior, credential type
  - Behavior Risk: Volume anomaly, high-risk events, failures
  - Resource Risk: Sensitivity of affected compartments
  - Network Risk: External access, unusual ports, traffic volume




RISK SCORE BREAKDOWN:
  Identity Risk:   2/7   🟢
    - API Key usage: False
    - First-time behavior: False
    - External IP access: True
    
  Behavior Risk:   8/8   🔴
    - Volume anomaly (z=4.8): True
    - Failed events: 20081
    - High-risk event chain: True
    
  Resource Risk:   0/6   🟢
    - Production access: False
    - Security compartment: False
    
  Network Risk:    1/5   🟢
    (Requires flow log correlation)

  TOTAL RISK:      11/26   🟡 MEDIUM


PHASE 10: INVESTIGATION SUMMARY

TARGET EVENT:          CreatePrivateIp
PARTITION DATE:        2026-01-28
TOTAL EVENTS:          20,146

ACTORS IDENTIFIED:
  Principals:          7
  Unique IPs:          7
  
KEY FINDINGS:
  New Behavior:        0 principals doing this for FIRST TIME
  Volume Z-Score:      4.76 (ANOMALY)
  Failed Events:       20081
  High-Risk Events:    26
  Production Impact:   0 events in prod compartments

RISK ASSESSMENT:       11/26 - 🟡 MEDIUM PRIORITY

RECOMMENDED ACTIONS:


🟡 MEDIUM PRIORITY - IN